# 📐 Module 3 — Class 2: Make Numbers Comparable (Scaling)

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/2](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/2)

---

## 📖 Today's story

Yesterday you **cleaned** the customer data. ✅ 

Today the boss asks: *"OK, train the model now!"*

But there is one more problem hiding in the data...

Look at our customers. We have:
- `tenure` → numbers from **0 to 72** (months)
- `MonthlyCharges` → numbers from **18 to 118** (dollars)
- `TotalCharges` → numbers from **0 to 8,500** (dollars)

All are numbers. **But the sizes are very different!**

The model will think: *"TotalCharges is most important because it's the biggest!"* ❌

**That is wrong.** A 1-month tenure change is NOT less important than a 1-dollar TotalCharges change. They just use different scales.

**Today we make all numbers fair to compare.**

---

## 💡 What is "scaling"?

**Scaling = making different numbers fit in a similar range.**

Example:
- BEFORE: tenure = 36 months, charges = 4,500 dollars (very different sizes)
- AFTER:  tenure = 0.5, charges = 0.5 (now they are equal in size)

We do not change WHAT the data means. We only change the **size of the numbers**.

---

## 🎯 Today's plan

1. **Setup** — get our cleaned data
2. **See the problem** — why different sizes break the model
3. **Tool 1: StandardScaler** — make mean = 0, spread = 1
4. **Tool 2: MinMaxScaler** — squeeze to [0, 1]
5. **Tool 3: RobustScaler** — best when outliers exist
6. **The BIG rule** — fit on train data ONLY (or you cheat!)
7. **Compare** — which scaler wins on a real model?
8. **Save** scaled data for Class 3

## 🤖 How to run

Click a code cell → press **Shift + Enter**.

👉 Each cell prints **what to look at** AND **why it matters**. Read the output text!

---

# 🧰 First — What is scikit-learn?

Today is the **first time** we use a new library: **scikit-learn** (people say *"sklearn"*).

Before we use it, let's understand what it is. This will take ~20 minutes but it will make EVERY future class easier.

## 📦 What it is

**scikit-learn = the standard ML library in Python.**

In Module 2 you learned **pandas** for data tables. Now learn **sklearn** for machine learning.

It contains hundreds of ready-made tools:
- 📐 tools to **prepare data** (scaling, encoding, cleaning numbers)
- 🤖 tools to **train models** (KNN, Random Forest, Linear Regression)
- 📊 tools to **check models** (accuracy, precision, recall)
- 🎯 tools to **find best settings** (auto-tuning)

You pick the right tool for your job.

## 💡 Why do we need sklearn? Why not write our own code?

If we wrote everything ourselves:
- ❌ Hundreds of lines of math for ONE model
- ❌ Slow (Python is not fast for math)
- ❌ Bugs! Math errors are very hard to find

With sklearn:
- ✅ 2-3 lines per model
- ✅ Fast (written in C under the hood)
- ✅ Tested by millions of developers — battle-tested code

**Almost EVERY ML project in the world uses sklearn. It is the industry standard.**

## 🌍 Real-world fact

In a job interview for an ML or data role:
- They will ask: *"Do you know sklearn?"*
- They will NOT ask: *"Can you write KNN from scratch?"*

So learning sklearn = learning a real job skill.

---

## 🎯 The 4 jobs sklearn does

Every ML project follows the same 4 steps. sklearn has tools for all of them.

| # | Job | What it means | Example tool |
| --- | --- | --- | --- |
| 1 | **Prepare data** | Make data ready for the model | `StandardScaler` |
| 2 | **Split data** | Train + test parts | `train_test_split` |
| 3 | **Train a model** | Teach the model to predict | `KNeighborsClassifier` |
| 4 | **Check the model** | Measure how good it is | `accuracy_score` |

We will use ALL 4 today!

## 💡 The great thing about sklearn — every tool works the same way

In other libraries, every tool has its own commands you must memorize separately.

**sklearn is different.** Every tool follows ONE of two simple patterns. Once you learn the pattern, ALL tools become easy.

Let me explain the 2 patterns 👇

---

## 🛠️ Pattern 1: Preprocessor tools (`fit` then `transform`)

These are tools that **prepare** the data. Like StandardScaler, MinMaxScaler, OneHotEncoder.

They follow 2 steps:

### Step 1: `fit()` — LEARN from the data

The tool looks at the data and **memorizes important numbers**.

For StandardScaler, it memorizes:
- The **mean** (average)
- The **std** (spread)

It does NOT change the data yet. Just learns.

```python
scaler = StandardScaler()
scaler.fit(X_train)   # ← learns mean and std from X_train
```

### Step 2: `transform()` — APPLY what was learned

Now use the memorized numbers to actually change the data.

```python
X_scaled = scaler.transform(X_train)   # ← uses the memorized mean/std to scale
```

### 🎁 Shortcut: `fit_transform()` does BOTH at once

If you want to fit and transform on the same data:

```python
X_scaled = scaler.fit_transform(X_train)   # ← fit AND transform in one line
```

**WARNING:** Use `fit_transform` ONLY on training data. Use `transform` (no fit) on test data!

### 💡 Why two separate steps?

Because we want to **fit ONCE** on training data, then apply the SAME numbers to many new datasets later (test data, future production data, new customers tomorrow).

We learn from training only — and apply forever.

`fit` = study the data and remember key numbers.
`transform` = process the data using those remembered numbers.

Two steps so you can study once, apply many times.

---

## 🤖 Pattern 2: Model tools (`fit` then `predict` then `score`)

These are tools that **predict** something. Like KNN, Random Forest, Logistic Regression.

They follow 3 steps:

### Step 1: `fit()` — LEARN from training data

The model looks at past customers and learns the patterns.

```python
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)   # ← learns from train data + correct answers
```

Notice: this `fit` takes **two** things:
- `X_train` = the features (numbers about each customer)
- `y_train` = the correct answer for each customer (did they leave? Yes/No)

### Step 2: `predict()` — GUESS the answer for new data

```python
predictions = model.predict(X_test)   # ← guesses Yes/No for each test customer
```

### Step 3: `score()` — CHECK how often the model is right

```python
accuracy = model.score(X_test, y_test)   # ← compares guesses with real answers
```

### 💡 Why three separate steps?

In real life, you train ONCE on past data, then **use the trained model many times**:
- Every new customer → call `predict()` to get a Yes/No
- Every month → call `score()` to check if the model is still accurate

`fit` once. `predict` for every new customer that arrives. `score` when you want to know how reliable the model still is.

**That's it!** Two patterns. Every sklearn tool uses one of them.

---

## 🎬 Hello sklearn — let's actually USE it

Before we touch our customer data, let's run a **tiny example** so you SEE the patterns work.

The next cell:
1. Takes 5 simple numbers: `[10, 20, 30, 40, 50]`
2. Uses StandardScaler to scale them
3. Shows what changed

This is the simplest possible sklearn example. After this, the rest of the lesson will feel easy.

In [ ]:
# 🎬 HELLO SKLEARN — the simplest possible example

# Step 1: import the tool we want to use from the sklearn library
from sklearn.preprocessing import StandardScaler

# Step 2: prepare a tiny dataset — just 5 simple numbers
# (Note the [[ ]] — sklearn always wants a 2D table format, not a 1D list,
#  so each number is wrapped in its own list to make a "column" of values)
data = [[10], [20], [30], [40], [50]]

print('🔢 BEFORE scaling:')
print(f'   Numbers: {[row[0] for row in data]}')   # show the original numbers
print(f'   Mean   : 30.0')                          # average of 10..50 is 30
print(f'   Range  : 10 to 50')                      # the spread of values
print()

# Step 3: create the StandardScaler tool (no settings needed for the default)
scaler = StandardScaler()

# Step 4: fit + transform in ONE line (the all-in-one shortcut)
#   - fit()       = look at the data, learn its mean and std
#   - transform() = use those learned numbers to rescale every value
#   fit_transform = both steps combined, only safe on TRAINING data
scaled = scaler.fit_transform(data)

print('✅ AFTER scaling:')
print(f'   Numbers: {[round(row[0], 2) for row in scaled]}')   # the new scaled values
print(f'   Mean   : {scaled.mean():.2f}  ← became 0!')          # mean is now exactly 0
print(f'   Std    : {scaled.std():.2f}  ← became 1!')           # std is now exactly 1
print()
print('━' * 50)
print('💡 What happened?')
print('   • fit() learned: mean=30, std=14.14')
print('   • transform() applied: new = (old - 30) / 14.14')
print('   • Result: mean=0, std=1 (the StandardScaler effect)')
print('━' * 50)

---

## 📦 What we'll import today (and why)

Now you know what sklearn is. Below are the 4 tools we'll import for today's lesson:

| Import | What it does | Pattern |
| --- | --- | --- |
| `StandardScaler` | Scale: mean=0, std=1 | Preprocessor (fit + transform) |
| `MinMaxScaler` | Scale: range [0, 1] | Preprocessor (fit + transform) |
| `RobustScaler` | Scale: uses median + IQR | Preprocessor (fit + transform) |
| `train_test_split` | Split data into 2 parts | Function (not a tool to fit) |
| `KNeighborsClassifier` | Predict using K nearest customers | Model (fit + predict + score) |
| `accuracy_score` | Measure: % of correct guesses | Function (compare 2 lists) |

You'll see all 6 today. **You already used StandardScaler above** — congratulations, you're using sklearn!

🎯 **Now we're ready.** Let's start the real lesson with our customer data 👇

---
## Setup

In [ ]:
# pandas — for tables of data (DataFrames). You used this in Module 2.
import pandas as pd
# numpy — for fast number arrays and math (mean, std, percentile, etc.)
import numpy as np
# matplotlib — for plotting charts (histograms, box plots, etc.)
import matplotlib.pyplot as plt
# Hide library deprecation warnings so the notebook output stays clean.
import warnings; warnings.filterwarnings('ignore')

# Today's stars — the 3 sklearn scalers we will compare side by side.
# All 3 follow the same Pattern 1: fit() learns, transform() applies.
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# train_test_split — splits the data into "learn from this" and "evaluate on this".
from sklearn.model_selection import train_test_split

# KNeighborsClassifier — our example model: predicts churn by looking at K nearest customers.
from sklearn.neighbors import KNeighborsClassifier

# accuracy_score — measures the % of predictions that matched the real answer.
from sklearn.metrics import accuracy_score

print('✅ Tools ready! We are using:')
print('   • pandas, numpy, matplotlib (you know these from Module 2)')
print('   • sklearn = scikit-learn (the BIG ML library)')

---
## Step 1: Get our cleaned data 📥

We need the file we cleaned in Class 1. Just to be safe, we re-do the C1 fixes here so you can run this notebook from scratch.

In [ ]:
# Re-load the Telco data and re-apply the 2 fixes from Class 1.
# (We do this here so this notebook works even if you skipped saving C1's output.)

# 1) Download the public Telco Customer Churn CSV from IBM's GitHub.
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)   # df is now a DataFrame with 7043 customers

# Fix 1: TotalCharges arrived as TEXT (with some empty strings for new customers).
# pd.to_numeric converts it to a real number column.
# errors='coerce' turns anything that fails to convert into NaN (missing).
# fillna(0) replaces those NaN values with 0 (the customer is brand new = no charges yet).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Fix 2: Telco's service columns have THREE values: 'Yes', 'No', 'No internet service'.
# 'No internet service' is just a fancy way of saying No, so we collapse it to 'No'.
# This makes encoding simpler in Class 3.
for c in ['OnlineSecurity','OnlineBackup','DeviceProtection',
          'TechSupport','StreamingTV','StreamingMovies']:
    df[c] = df[c].replace('No internet service', 'No')

# Same idea for MultipleLines: 'No phone service' just means No.
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Quick sanity-check print so we know loading worked.
print(f'✅ Loaded {len(df)} customers from Telco. Ready to scale!')

---
## Step 2: See the problem 👀

> **Why do we need scaling?**

Let's look at 3 number columns and see how different their sizes are.

In [ ]:
# Goal: show the SIZE difference between our 3 number columns.
# These are the columns we will scale today.
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

print('🔍 Range of values for each column:')
print()

# Loop through each column and print its min and max value.
# .min() and .max() are pandas methods that return the smallest/largest number.
for c in cols:
    # f-string with formatting:  :18s = pad name to 18 chars, :>8.1f = number, 1 decimal.
    print(f'  {c:18s} : min = {df[c].min():>8.1f}  max = {df[c].max():>8.1f}')

print()
print('━' * 50)
print('🚨 PROBLEM: The 3 columns are on VERY different scales!')
print()
print('   tenure        goes from 0 to ~72       (small numbers)')
print('   MonthlyCharges goes from ~18 to ~118    (medium numbers)')
print('   TotalCharges  goes from 0 to ~8,500    (BIG numbers)')
print()
print('💡 The model will think TotalCharges is the "most important"')
print('   feature — just because the numbers are bigger.')
print('   This is WRONG. Big number ≠ more important.')
print('━' * 50)

### 🤔 Why does this happen?

Many ML models use **distance** to compare customers.

**Distance** = how far apart 2 customers are. Like measuring 2 cities on a map.

Imagine 2 customers:
- Ali: tenure = 10, MonthlyCharges = 30, TotalCharges = 300
- Zara: tenure = 12, MonthlyCharges = 32, TotalCharges = 400

Difference:
- tenure: 2 (small)
- MonthlyCharges: 2 (small)
- **TotalCharges: 100 (BIG!)**

The model sees: *"Ali and Zara are very different — TotalCharges differs by 100!"*

But really, all 3 differences mean the same thing! TotalCharges just LOOKS bigger because of the scale.

👉 **Scaling fixes this.** It makes a 2-month tenure difference equally important to a 200-dollar TotalCharges difference.

### Which models care about scaling?

| Model type | Cares? | Why |
| --- | --- | --- |
| 📏 **K-NN, K-Means** | ✅ YES | They use distance |
| 📈 **Linear / Logistic Regression** | ✅ YES | Math works better with similar sizes |
| 🧠 **Neural networks** | ✅ YES | Same — math works better |
| 🌳 **Decision Trees, Random Forest** | ❌ No | They split by 'is X > 5?' — size doesn't matter |

💡 **Pro tip:** When in doubt, scale. It rarely hurts trees, and it really helps everything else.

---

## 🎓 Why each model type cares (or doesn't)

The table above said WHICH models care about scaling. Now let's go through each row and understand **WHY**.

---

### 📏 K-NN and K-Means → cares because of **distance**

**What is K-NN?**
> *"For new customer Ali, find the **5 closest customers** in our data. They tell us if Ali will leave or stay."*

**What is K-Means?**
> *"Group similar customers together — like organising the customer base into segments based on how alike they are."*

**Why scale matters:**

Both models compute **distance** between customers — like how far apart 2 cities are on a map.

| Column | Ali | Zara | Difference |
| --- | --- | --- | --- |
| age | 30 | 35 | 5 |
| TotalCharges | 1000 | 5000 | **4000** |

The model only sees the numbers, not the meaning. So it thinks: *"Ali and Zara differ by 4,000 + 5 = 4,005. They are very different."*

But really, the 5-year age difference is just as important as the charges difference. The big number drowns out the small one.

**👉 Scaling fixes this** by putting both columns on the same range (e.g. -1 to +1). Then the model treats every column fairly.

---

### 📈 Linear and Logistic Regression → cares because **the math runs faster and more stable**

**What is Linear Regression?**
> *"Find the best straight line through the data."* (Predicts numbers like price, age.)

**What is Logistic Regression?**
> *"Find a line that separates Yes customers from No customers."* (Predicts Yes/No.)

**Why scale matters:**

The model finds the best line by adjusting **weights** (one number per column). It does this with many small repeated updates.

**Without scaling:**
- TotalCharges has values up to 8,500 → its weight needs tiny updates
- tenure has values up to 72 → its weight needs much bigger updates
- The math has to balance updates that differ by 100×
- Result: training is slow, sometimes the model never settles on an answer

**With scaling:**
- Every column lives in the same range
- All weights update at similar rates
- Training is fast and stable

---

### 🧠 Neural Networks → cares because **big numbers break the math**

**What is a Neural Network?**
> *"Many small computing units (called neurons) connected together. Each one does math: takes a number in, gives a number out."*

**Why scale matters:**

Each neuron does: `output = (input × weight) + bias`

- If `input = 8500` (TotalCharges) → output is huge → the math becomes unstable (numbers overflow, gradients explode)
- If `input = 0.001` → output is tiny → no useful signal → model can't learn from this column

After scaling, every input lives in a normal range (around -1 to +1). The math stays stable, and every column contributes a real signal to learning.

---

### 🌳 Decision Trees and Random Forest → DOES NOT care

**What is a Decision Tree?**
> *"A flowchart of yes/no questions. 'Is age > 30? YES → ... NO → ...'"*

**What is a Random Forest?**
> *"Many different trees, all voting on the answer."*

**Why scale doesn't matter:**

Trees only ask **threshold questions**:
- *"Is tenure > 12 months?"* ✅ or ❌
- *"Is monthly_charges > 50?"* ✅ or ❌

The answer is the same whether you measure in months or seconds, dollars or cents. The tree finds the best threshold by itself, regardless of scale.

(Quick way to remember: scaling shifts and shrinks all values together. The relative order of values stays the same — and that's all a tree cares about.)

---

### 💡 The "WHEN IN DOUBT" rule

> *"If you're not sure if a model needs scaling — just scale."*

- ✅ It almost never hurts trees
- ✅ It dramatically helps everything else
- ✅ Scaling costs you 1 extra line of code

---

### 🗺️ Where you'll see these models again

| Module | Class | Model | Needs scaling? |
| --- | --- | --- | --- |
| **M4** | Class 1 | Linear Regression | ✅ YES |
| **M4** | Class 2 | Logistic Regression | ✅ YES |
| **M4** | Class 3 | Decision Trees + Random Forest | ❌ No |
| **M4** | Class 4 | KNN + SVM | ✅ YES (distance!) |
| **M5** | Class 1 | K-Means | ✅ YES (distance!) |
| **M6** | All | Neural Networks | ✅ YES |

So today's lesson is the **foundation** for the next 6 weeks of teaching!

---
## Step 3: Tool 1 — StandardScaler 📊

### What is it?

**StandardScaler** = the most popular scaler. It changes each number so:
- The **mean** (average) becomes **0**
- The **spread** (standard deviation) becomes **1**

### Why do we need it?

It puts all columns on the same "playing field". After StandardScaler, every column has the same mean and spread. The model can compare them fairly.

### How does it work? (the simple math)

For each value: `new_value = (old_value - mean) / standard_deviation`

**In words:** *"How many standard deviations is this value from the average?"*

Don't worry about the math — sklearn does it for us!

### When to use?

✅ **Default choice** — use it when you don't know what to do.
✅ Good for most models (Logistic Regression, KNN, Neural Networks).
❌ Bad when there are huge outliers (use RobustScaler instead — see Step 5).

---

### ⚠️ Hold on! Why do we see NEGATIVE numbers after scaling?

When you run StandardScaler, you'll see numbers like `-1.32`, `-0.91`, `+1.61`. 

**What do these mean? Let's figure it out together with REAL math.**

### 🧮 The formula in 3 simple words

```
new_value = (old_value - mean) / standard_deviation
```

In plain English:
> *"How many **steps** away is this value from the average?"*

Where each "step" = one **standard deviation** (the typical spread).

### What the sign tells us

| Sign | Meaning |
| --- | --- |
| **POSITIVE** (e.g. +1.61) | Value is **ABOVE** the average |
| **ZERO** (0.00) | Value is **EQUAL** to the average |
| **NEGATIVE** (e.g. -1.32) | Value is **BELOW** the average |

### What the size tells us

| Size | Meaning |
| --- | --- |
| Between -1 and +1 | **Normal** customer (close to average) |
| Beyond ±2 | **Unusual** customer (far from average) |
| Beyond ±3 | **Very rare** customer (might be outlier!) |

### 🎯 Think of it like this

A customer with `tenure = 72 months` becomes `+1.61`. This means:
> *"This customer's tenure is **1.61 steps ABOVE** the average customer."*

A customer with `tenure = 0 months` becomes `-1.32`. This means:
> *"This customer's tenure is **1.32 steps BELOW** the average customer."*

The numbers are not random! Let's calculate them BY HAND in the next cell.

### 🤔 Wait — what is "mean" and "std" again?

Before we believe the math, let's make sure we know these 2 simple words.

**Mean** = the average of all values.
- Example: ages of 5 students: [10, 12, 14, 16, 18]
- Mean = (10+12+14+16+18) / 5 = **14**

**Standard deviation (std)** = how SPREAD OUT the values are from the mean.
- Small std → values are CLOSE to the mean (like [13, 14, 14, 14, 15] → small std)
- Big std → values are FAR from the mean (like [5, 10, 14, 18, 23] → big std)
- It's the "typical distance" from the mean

**For our `tenure` column in Telco:**
- mean ≈ 32.37 → average customer has been with us about 32 months
- std ≈ 24.56 → typical customer is about 24 months away from average (some have 8 months, some have 56 months — that's the typical spread)

OK — now let's run the math for real customers 👇

In [ ]:
# 🧮 Let's do the math BY HAND on real customers from our dataset.
# Goal: prove that the negative numbers from StandardScaler are not random —
# they come from a simple 2-step formula we can compute ourselves.

# Step 1: compute the mean (average) and std (typical spread) of the tenure column.
# These are the 2 numbers StandardScaler will memorize when we call fit().
mean_t = df['tenure'].mean()   # average tenure across all 7043 customers
std_t  = df['tenure'].std()    # standard deviation = typical distance from the mean

print('🔢 For the tenure column in our REAL Telco data:')
print(f'   mean = {mean_t:.2f}  ← average tenure across all 7043 customers')
print(f'   std  = {std_t:.2f}  ← typical spread (one "step")')
print()
print('━' * 55)
print('📐 The formula:  new = (old - mean) / std')
print('━' * 55)
print()

# Step 2: pick 4 example tenure values that span the range.
# Each tuple is (raw_tenure_value, label_describing_this_customer).
examples = [
    (72, 'A LOYAL customer (longest tenure)'),
    (32, 'An AVERAGE customer'),
    (10, 'A NEW-ish customer'),
    (0,  'A BRAND NEW customer (just signed up)')
]

# Walk through the formula step by step for each example.
for val, label in examples:
    # Apply the StandardScaler formula manually.
    new_val = (val - mean_t) / std_t

    # Decide if the result is above, below, or equal to the average.
    sign_word = 'ABOVE' if new_val > 0 else ('BELOW' if new_val < 0 else 'EQUAL TO')

    # Print each step so the math is visible — no "magic", just arithmetic.
    print(f'👤 {label}: tenure = {val}')
    print(f'   Step 1: subtract the mean')
    print(f'           {val} - {mean_t:.2f} = {val - mean_t:+.2f}')
    print(f'   Step 2: divide by the std')
    print(f'           {val - mean_t:+.2f} / {std_t:.2f} = {new_val:+.2f}')
    print(f'   ✨ Result: {new_val:+.2f}  ({sign_word} the average)')
    print()

print('━' * 55)
print('💡 NOW you understand!')
print('   • Negative numbers = customer is BELOW average tenure')
print('   • Positive numbers = customer is ABOVE average tenure')
print('   • Zero = customer is EXACTLY the average')
print('   • The size (1.32, 1.61) = how many "steps" away')
print('━' * 55)

---

## 🎯 So why does this matter for the model?

You just saw the math. Now let's connect it back to **why** we did all this.

### Before scaling — 3 columns measured in 3 different units

| Column | Range | Unit |
| --- | --- | --- |
| tenure | 0 to 72 | months |
| MonthlyCharges | 18 to 118 | dollars per month |
| TotalCharges | 0 to 8,500 | dollars total |

The model doesn't see the units — it sees only the raw numbers. So it thinks: *"TotalCharges goes up to 8,500. That column must matter most."* ❌

### After StandardScaler — every column on the same scale

| Column | New range | Typical value |
| --- | --- | --- |
| tenure | -1.32 to +1.61 | between -1 and +1 |
| MonthlyCharges | -1.55 to +1.79 | between -1 and +1 |
| TotalCharges | -1.01 to +2.83 | between -1 and +1 |

Now the model sees: *"All 3 columns sit in the same range. I will compare them fairly."* ✅

### 🎓 Three things to remember

1. **Negative ≠ bad.** Negative just means *"below the average."* Perfectly normal value.
2. **Zero = average customer.** A scaled value of 0 means exactly the column average.
3. **Size = how unusual.** The further a number is from 0, the more unusual that customer is on this column.

Now we are ready to apply StandardScaler to all 3 columns at once 👇

In [ ]:
# Apply StandardScaler to all 3 columns at once — this is what we just did by hand,
# but on every row in the dataset.

# Step 1: create the scaler tool (default settings: mean=0, std=1).
scaler = StandardScaler()

# Step 2: pick the 3 number columns from our DataFrame.
# X is now a smaller DataFrame with just tenure, MonthlyCharges, TotalCharges.
X = df[cols]

# Step 3: fit_transform = "learn the mean+std for each column, then apply the formula".
# Returns a numpy array (not a DataFrame) — sklearn always returns arrays.
X_standard = scaler.fit_transform(X)

# Wrap the array back into a DataFrame so we can use .describe() and column names.
X_standard_df = pd.DataFrame(X_standard, columns=cols)

print('✅ StandardScaler applied!')
print()
print('🔍 BEFORE (original values):')
# .describe() gives us a stats summary. We pick out 4 useful rows to show.
# .round(2) keeps 2 decimal places for readability.
print(X.describe().loc[['mean', 'std', 'min', 'max']].round(2))
print()
print('🔍 AFTER StandardScaler:')
print(X_standard_df.describe().loc[['mean', 'std', 'min', 'max']].round(2))
print()
print('━' * 50)
print('👉 What changed:')
print('   • mean is now ~0 (was 32.4 / 64.8 / 2280)')
print('   • std is now ~1 (was 24.6 / 30.1 / 2266)')
print('   • All 3 columns are now on the SAME scale!')
print('━' * 50)

---
## Step 4: Tool 2 — MinMaxScaler 📐

### What is it?

**MinMaxScaler** = squeezes all values into a fixed range, usually **[0, 1]**.

- The smallest value → becomes 0
- The biggest value → becomes 1
- Everything else → spread between 0 and 1

### Why use it?

Some models REQUIRE inputs in [0, 1]. Examples:
- Neural networks with sigmoid activation
- Image processing (pixel values 0-1)

### How it works

`new_value = (old_value - min) / (max - min)`

### When to use?

✅ When the model needs values in [0, 1]
✅ When data has NO big outliers
❌ **Dangerous with outliers** — see what happens below!

In [ ]:
# Apply MinMaxScaler — squeezes every column into the range [0, 1].
# Formula: new = (old - min) / (max - min)
#   - the smallest value in the column becomes 0
#   - the biggest value in the column becomes 1
#   - everything else lands somewhere between

# Step 1: create the tool (default range is [0, 1]).
scaler = MinMaxScaler()

# Step 2: fit_transform — learn the min/max from the data, then apply the formula.
X_minmax = scaler.fit_transform(X)

# Step 3: wrap the array back into a DataFrame for nicer display.
X_minmax_df = pd.DataFrame(X_minmax, columns=cols)

print('✅ MinMaxScaler applied!')
print()
print('🔍 AFTER MinMaxScaler:')
# Show only min, max, and mean — the 3 stats that prove the squeeze worked.
print(X_minmax_df.describe().loc[['min', 'max', 'mean']].round(3))
print()
print('━' * 50)
print('👉 Notice: min = 0.0 and max = 1.0 for ALL columns!')
print('   Everything is now squeezed into [0, 1].')
print('━' * 50)

### 🚨 The MinMax outlier trap

Imagine ONE customer pays $99,999 (typo or fraud). What happens?

MinMax sees: max = 99,999. So it makes 99,999 → 1.0.

But everyone else (paying $20–$120) gets squeezed into a TINY space at the bottom — like 0.0001 to 0.0012!

**Result:** 99% of your data uses only 0.1% of the [0, 1] range. Most info is lost.

**Lesson:** Never use MinMaxScaler if you have wild outliers. Clean outliers FIRST, or use RobustScaler instead.

In [ ]:
# DEMO: prove that ONE outlier completely breaks MinMaxScaler.

# Step 1: copy the data so we don't ruin the real X.
# .copy() is important — without it, edits would change the original X too.
X_with_outlier = X.copy()

# Step 2: inject ONE crazy value into the first row's TotalCharges.
# 99,999 simulates a typo, fraud, or data-entry error.
X_with_outlier.loc[0, 'TotalCharges'] = 99999

# Step 3: fit a fresh MinMaxScaler on this tainted data and transform it.
scaler = MinMaxScaler()
X_outlier_scaled = scaler.fit_transform(X_with_outlier)
outlier_df = pd.DataFrame(X_outlier_scaled, columns=cols)

print('🚨 DEMO: What ONE outlier does to MinMax')
print()
print('TotalCharges values after MinMax (with one 99999 outlier):')
# loc[0, 'TotalCharges'] = the outlier row → MinMax assigns it 1.0 (the max).
print(f'   The outlier row    : {outlier_df.loc[0, "TotalCharges"]:.4f}')
# .quantile(0.95) = the value at the 95th percentile (95% of customers are below this).
print(f'   95% of customers : {outlier_df["TotalCharges"].quantile(0.95):.4f}')
# .quantile(0.5) = the median (50th percentile).
print(f'   50% of customers : {outlier_df["TotalCharges"].quantile(0.5):.4f}')
print()
print('━' * 50)
print('👉 99,999 → 1.0 (the max)')
print('   95% of REAL customers got squeezed into [0, 0.085]')
print('   That is only 8.5% of the [0,1] space!')
print('   Most of our information is LOST.')
print('━' * 50)

---

## 🛑 STOP — before we meet RobustScaler, what is an outlier?

You will hear this word A LOT in ML. Let's understand it first.

### Outlier = a number that is FAR away from the rest

Imagine you measure the height of 10 people in Tashkent:

| Person | Height (cm) |
| --- | --- |
| 1 | 170 |
| 2 | 165 |
| 3 | 172 |
| 4 | 168 |
| 5 | 175 |
| 6 | 169 |
| 7 | 171 |
| 8 | 167 |
| 9 | 173 |
| 10 | **🚨 350** ← OUTLIER |

Person 10 has height = 350 cm. That is 3.5 meters tall. **Impossible for a human.**

It's probably:
- A **typo** (someone wrote 350 instead of 170)
- A **measurement mistake** (broken sensor)
- A **data entry error** (extra zero added)
- Or one truly rare strange case (1 in a million event)

### Picture in your head

```
the typical values                              the outlier
●●●●●●●●●                                          🚨
165  170  175 ............................... 350
←——— most data lives here ———→             ←— far away —
```

The outlier is the value that sits far from the rest of the data.

### Where outliers come from in our Telco / Tashkent data

- A customer with `TotalCharges = 99,999` dollars (typo or fraud)
- A customer with `tenure = 999` months (~83 years — impossible)
- A real billionaire who pays 50× more than everyone (rare but real)
- A bug in the billing system that put 0 instead of the real value

Some are mistakes. Some are real-but-rare. Either way → they make the math go crazy.

### Why is this important for scaling?

In the next cell, we'll see with our own eyes what ONE outlier does to:
- **mean** and **std** (used by StandardScaler) → completely **ruined**
- **median** and **IQR** (used by RobustScaler) → barely move

That is the WHOLE reason RobustScaler exists. Let's prove it 👇

In [ ]:
# 🧪 Watch what an outlier does to mean/std vs median/IQR.
# This is the proof that explains WHY RobustScaler exists.

# Two datasets: 10 normal customer values, and the same 10 + 1 outlier of 9999.
normal   = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65]
with_bad = normal + [9999]

# np.percentile(data, [25, 75]) returns Q1 and Q3 — the cuts at 25% and 75%.
# IQR = Q3 - Q1 (the spread of the middle half of the data).
q1_n, q3_n = np.percentile(normal,   [25, 75])
q1_b, q3_b = np.percentile(with_bad, [25, 75])

# Print the 4 stats for the clean data first (baseline).
print('🔍 WITHOUT outlier (10 normal customers):')
print(f'   mean   = {np.mean(normal):>7.1f}')           # average
print(f'   std    = {np.std(normal):>7.1f}')            # standard deviation
print(f'   median = {np.median(normal):>7.1f}')         # middle value
print(f'   IQR    = {q3_n - q1_n:>7.1f}')               # middle 50% spread
print()

# Now print the same 4 stats AFTER injecting the outlier — see who survives.
print('💥 WITH ONE outlier (10 normal + 9999):')
print(f'   mean   = {np.mean(with_bad):>7.1f}    ← went CRAZY (~10x bigger!)')
print(f'   std    = {np.std(with_bad):>7.1f}    ← went CRAZY (~200x bigger!)')
print(f'   median = {np.median(with_bad):>7.1f}    ← barely moved 👍')
print(f'   IQR    = {q3_b - q1_b:>7.1f}    ← barely moved 👍')
print()
print('━' * 55)
print('💡 LESSON:')
print('   • mean and std are EASILY ruined by outliers')
print('   • median and IQR barely change — they are STRONG')
print()
print('   → StandardScaler uses mean/std → gets ruined by outliers')
print('   → RobustScaler uses median/IQR → stays safe!')
print('━' * 55)

---

## 📏 What is "median" and "IQR"?

These 2 simple words = the **secret** behind RobustScaler.

### Median = the MIDDLE value (when data is sorted)

Take 7 numbers, sort them, pick the middle one:

```
[3, 5, 8, 12, 18, 22, 30]
        ↑
     median = 12
```

Now change the highest number from 30 to 99999:

```
[3, 5, 8, 12, 18, 22, 99999]
        ↑
     median is STILL 12  ← outlier did nothing!
```

**Outliers cannot move the median.** ✊ That is the property we want.

### Quartiles — split sorted data into 4 equal groups

Imagine sorting 100 customers from smallest to biggest. Cut them into 4 equal groups of 25 each:

```
[smallest 25%] | [next 25%] | [next 25%] | [biggest 25%]
                ↑            ↑            ↑
                Q1        median (Q2)     Q3
              (25% mark)   (50% mark)   (75% mark)
```

- **Q1** (first quartile) = the value at the **25%** mark
- **Q2** = the **median** (50% mark)
- **Q3** (third quartile) = the value at the **75%** mark

### IQR = Q3 − Q1 = the middle 50% range

**IQR** stands for **Inter-Quartile Range**. It tells us:

> *"How spread out is the **middle half** of the data?"*

```
   ←——— IQR (middle 50%) ———→

[smallest] | Q1 ●●●●●● median ●●●●●● Q3 | [biggest]
            ↑                          ↑
        cut off here              cut off here
        (ignore outliers)         (ignore outliers)
```

**Why is this useful?** We **ignore** the smallest 25% and biggest 25%. Outliers (which sit at the extreme ends) are cut off and can't break the math.

### Real example

Customer ages: `[20, 22, 25, 28, 30, 32, 35, 40, 99999]`

- **Median** = 30 (middle value, ignores the 99999)
- **Q1** = 24 (one quarter from bottom)
- **Q3** = 36 (three quarters from bottom)
- **IQR** = 36 − 24 = **12** ← the middle half is 12 wide

Even with 99999 in there, IQR is just 12. **Outlier-proof!** 🛡️

Now you understand what RobustScaler is doing under the hood: it uses *(value − median) / IQR* instead of *(value − mean) / std*. Same idea, but with outlier-proof tools.

---
## Step 5: Tool 3 — RobustScaler 🛡️

### What is it?

**RobustScaler** = a scaler that is **robust** (= strong, not affected) to outliers.

Instead of using mean and std (which outliers ruin), it uses:
- The **median** (middle value)
- The **IQR** (the middle 50% range — Q3 minus Q1)

### Why it works

The median doesn't change if there's an outlier. The IQR doesn't change either. So a few crazy values can NOT mess up the scaling.

### When to use?

✅ When data has outliers you can't easily remove
✅ Skewed distributions (values lean to one side)
✅ Telecom / banking / fraud data — always has outliers

In [ ]:
# Apply RobustScaler to the SAME tainted dataset that broke MinMaxScaler.
# Goal: prove RobustScaler keeps the real customers in a normal range
# even when the data contains a wild outlier.

# Step 1: create the RobustScaler tool.
# Internally it uses (value - median) / IQR instead of (value - mean) / std.
scaler = RobustScaler()

# Step 2: fit + transform on the data that has the 99999 outlier injected.
X_robust = scaler.fit_transform(X_with_outlier)

# Step 3: wrap into a DataFrame for easy column-by-column inspection.
robust_df = pd.DataFrame(X_robust, columns=cols)

print('🛡️ RobustScaler applied (to data WITH the 99999 outlier):')
print()
print('TotalCharges values after RobustScaler:')
# The outlier row — should be a big number, but not ridiculous.
print(f'   The outlier row     : {robust_df.loc[0, "TotalCharges"]:.2f}')
# 95th percentile — most of the real customers are below this.
print(f'   95% of customers  : {robust_df["TotalCharges"].quantile(0.95):.2f}')
# Median — should be very close to 0 because RobustScaler centers on the median.
print(f'   50% of customers  : {robust_df["TotalCharges"].quantile(0.5):.2f}')
print()
print('━' * 50)
print('👉 The outlier is now ~22 (still big — that is correct!)')
print('   But REAL customers get a NORMAL spread (around 0).')
print('   No information lost!')
print('   This is why RobustScaler wins when outliers exist.')
print('━' * 50)

---
## Step 6: Compare all 3 scalers side by side 📊

In [ ]:
# Visual side-by-side comparison: take ONE column (MonthlyCharges) and show
# how each scaler reshapes it. The shape stays the same; only the x-axis changes.

# Create 1 row of 4 charts (1x4 grid). figsize is in inches.
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))

# 4 versions of MonthlyCharges to plot:
#   1. Original          — raw data, no scaling at all.
#   2. StandardScaler    — recomputed earlier, mean=0, std=1.
#   3. MinMaxScaler      — recomputed earlier, range [0, 1].
#   4. RobustScaler      — fit fresh here on clean X (no outlier injected).
data_views = [
    ('Original',       df['MonthlyCharges']),
    ('StandardScaler', X_standard_df['MonthlyCharges']),
    ('MinMaxScaler',   X_minmax_df['MonthlyCharges']),
    ('RobustScaler',   pd.DataFrame(RobustScaler().fit_transform(X), columns=cols)['MonthlyCharges'])
]

# Loop through each version and draw a histogram.
for i, (name, data) in enumerate(data_views):
    # bins=30 = split the value range into 30 equal-width buckets.
    ax[i].hist(data, bins=30, color='steelblue', edgecolor='white')
    ax[i].set_title(name)            # chart title (which scaler)
    ax[i].set_xlabel('value')        # x-axis label
    if i == 0:
        ax[i].set_ylabel('count')    # only label y-axis on the leftmost chart

plt.suptitle('MonthlyCharges — same data, 4 different scales', fontsize=14)
plt.tight_layout()                   # prevent labels from overlapping
plt.show()

print()
print('━' * 50)
print('👉 What you see in each chart:')
print()
print('  1. Original       → x-axis: 18 to 118 (real dollars)')
print('  2. StandardScaler → x-axis: ~-2 to ~2 (mean=0, std=1)')
print('  3. MinMaxScaler   → x-axis: 0 to 1   (squeezed)')
print('  4. RobustScaler   → x-axis: ~-1 to ~1 (centered on median)')
print()
print('💡 The SHAPE is the same — only the SIZE changes!')
print('   Scaling does not change the data. It changes the scale.')
print('━' * 50)

---

## 🎁 Bonus — are these the ONLY scalers?

We learned **3** today: StandardScaler, MinMaxScaler, RobustScaler.

These are the **3 most popular**. But sklearn has more! Here's a fuller picture:

| Scaler | What it does | When to use |
| --- | --- | --- |
| **StandardScaler** ⭐ | mean=0, std=1 | **Default. Use this 80% of the time.** |
| **MinMaxScaler** | squeeze to [0, 1] | When the model needs values in [0,1] (neural nets, images) |
| **RobustScaler** 🛡️ | uses median + IQR | When data has wild outliers |
| **MaxAbsScaler** | divide by biggest absolute value | When data is sparse (lots of zeros) |
| **Normalizer** | makes each ROW sum to 1 | Text/document data — note: works on rows, not columns |
| **PowerTransformer** | makes data look like a bell curve | When data is very skewed |
| **QuantileTransformer** | maps to a normal distribution | Strong fix for weird-shaped data |

### 💡 You don't need to memorize all 7

**The simple rule of thumb:**

1. **First try:** `StandardScaler` ← works for ~80% of cases
2. If outliers exist → switch to `RobustScaler`
3. If model needs [0, 1] → switch to `MinMaxScaler`
4. The other 4? You'll meet them later when you actually need them.

### 🌍 In real ML jobs

- **StandardScaler** is the most common. **Memorize this one first.**
- **MinMaxScaler** for neural networks / image data (pixel values 0-255 → 0-1).
- **RobustScaler** for finance / banking / fraud / telecom — these fields ALWAYS have outliers.

These 3 cover 99% of real work. ✅

---
## Step 7: 🚨 The BIG RULE — fit on TRAIN data only

This is the **most important rule** of the whole module. Listen carefully.

### What is train/test split?

Before training a model, we split the data into 2 parts:
- **Train data** (~80%) — the model learns from this
- **Test data** (~20%) — we keep it separate to measure performance later

Why? Because we need an honest measurement of how the model performs on data it has **never seen before**. Test data is our reality check.

### What is data leakage?

**Data leakage = information from the test data secretly leaks into training.**

If the model has any information about the test data while learning, the test results are no longer an honest measurement. The model looks great on the test, but fails on real new customers in production.

This is the **#1 reason ML projects fail** when they go live.

The most common way leakage happens? **The scaler.** If you fit the scaler on the full dataset (train + test together), the scaler "saw" the test data:
- It used test values to compute the mean and std
- The training data was then scaled using info that included the test
- Now the test set is no longer truly unseen

### The wrong way ❌

```python
# WRONG! Scaler sees ALL data, including test
X_scaled = scaler.fit_transform(X)
X_train, X_test = train_test_split(X_scaled)
```

### The right way ✅

```python
# Step 1: split FIRST
X_train, X_test = train_test_split(X)

# Step 2: scaler learns from train ONLY
X_train_scaled = scaler.fit_transform(X_train)

# Step 3: same scaler applied to test (no learning, just apply)
X_test_scaled = scaler.transform(X_test)
```

### Why this matters in production

In real life, when a NEW customer arrives, you don't have their full history yet. You scale them using the **scaler that was trained months ago**. So the test data should be scaled the same way — using the train-time scaler, not a new one.

If your evaluation does something the production system can't do (like seeing future data), your evaluation is dishonest.

**Memorize:** *fit_transform on TRAIN, transform on TEST. Never both.*

In [ ]:
# Demonstrate the CORRECT order: split FIRST, then scale.

# Step 0: build the target column y.
# 'Churn' has values 'Yes' / 'No'. Convert to 1/0 so the model can use it.
# (== 'Yes') gives True/False, .astype(int) turns those into 1/0.
y = (df['Churn'] == 'Yes').astype(int)

# STEP 1: split the data into train (80%) and test (20%) BEFORE any scaling.
# random_state=42 fixes the random seed so we get the same split every run.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# STEP 2: create a fresh scaler and fit_transform ONLY on the training data.
# The scaler now knows the train mean/std — but has not seen the test data at all.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# STEP 3: apply (transform only — NO fit!) the same scaler to the test data.
# Test data is scaled with the train-time mean/std, just like real new customers
# would be scaled in production with a saved scaler.
X_test_scaled = scaler.transform(X_test)

print('✅ The right way!')
print(f'   Train rows: {len(X_train)}')   # ~5634 customers
print(f'   Test rows : {len(X_test)}')    # ~1409 customers
print()
print('   Train MonthlyCharges:')
# Before scaling, the column has its real-dollar mean (~64).
print(f'     Before scaling: mean = {X_train["MonthlyCharges"].mean():.2f}')
# After scaling, the column 1 (MonthlyCharges) mean should be ~0 — proof StandardScaler centered it.
print(f'     After scaling : mean = {X_train_scaled[:, 1].mean():.2f}  (should be ~0)')
print()
print('💡 The scaler ONLY saw training data when learning.')
print('   Test data was scaled using the same numbers, but never used for learning.')

---
## Step 8: Hands-on — which scaler wins on a real model? 🎯

Let's train a **K-Nearest Neighbors (KNN)** model to predict churn.

**KNN** = a model that finds the K closest customers to predict. Closest = uses **distance**. So it CARES about scaling a lot.

We'll try 4 versions:
- No scaling
- StandardScaler
- MinMaxScaler
- RobustScaler

Which one will give the best accuracy?

In [ ]:
# Train the SAME KNN model 4 times — once per scaler version — and compare accuracy.
# This is the bake-off: which scaler actually wins on a real model?

# Dictionary mapping a label → the scaler tool to use (None = no scaling).
# Using a dict lets us loop with both name and tool at once.
scalers = {
    'No scaling':     None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler()
}

print('🏁 Training KNN with each scaler...\n')

# We will collect each scaler's accuracy in this dict so we can compare at the end.
results = {}

for name, scaler in scalers.items():
    if scaler is None:
        # No-scaling case: just use the raw values.
        # .values converts the DataFrame into a numpy array (KNN expects arrays).
        Xtr, Xte = X_train.values, X_test.values
    else:
        # Scaling case: fit on train ONLY, transform both train and test.
        # This avoids data leakage (see the BIG RULE section).
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)

    # Create a new KNN model with k=5 (looks at 5 nearest customers to predict).
    knn = KNeighborsClassifier(n_neighbors=5)

    # Train (fit) the model on the scaled training data + correct answers.
    knn.fit(Xtr, y_train)

    # Predict on the test set, then compare predictions to the true labels.
    acc = accuracy_score(y_test, knn.predict(Xte))

    # Save the result and print a one-line summary.
    results[name] = acc
    print(f'   {name:18s} → {acc*100:.2f}% accuracy')

print()
print('━' * 50)

# Find the best and worst scaler from our results dict.
# max(dict, key=dict.get) returns the key with the highest value.
best  = max(results, key=results.get)
worst = min(results, key=results.get)

# Convert the accuracy gap to percentage points (×100 to read nicer).
diff = (results[best] - results[worst]) * 100

print(f'🏆 BEST : {best} ({results[best]*100:.2f}%)')
print(f'😢 WORST: {worst} ({results[worst]*100:.2f}%)')
print(f'   Difference: {diff:.1f} percentage points')
print()
print('💡 Scaling really matters for distance-based models like KNN!')
print('━' * 50)

### 🤔 Stop and think

1. By how much did the BEST scaler beat the WORST?
2. Why did `No scaling` fail badly?
3. Imagine 1000 customers and 10% accuracy difference. How many predictions are now correct that were wrong before?

---
## Step 9: Save the scaled data 💾

We will use this cleaned + scaled data in **Class 3** (feature engineering).

We pick the scaler that worked best (or most popular: StandardScaler) and save the result.

In [ ]:
# Save the scaled training data + the scaler object itself.
# Class 3 will load these files and continue from here.

# joblib is a small library for saving Python objects to disk.
# Used by sklearn for scalers, models, pipelines, etc.
import joblib

# Build a DataFrame from the scaled training data (numpy array → DataFrame).
out = pd.DataFrame(X_train_scaled, columns=cols)
# Attach the target column (Churn 1/0) so Class 3 has both X and y in one file.
out['Churn'] = y_train.values

# Save to CSV so it's human-readable and easy to load with pd.read_csv.
out.to_csv('telco_scaled.csv', index=False)

# Save the SCALER OBJECT to disk so the same mean/std can be reused later.
# In production, we would load this file, call .transform() on new customer data.
joblib.dump(scaler, 'scaler.joblib')

print('✅ Saved 2 files:')
print('   • telco_scaled.csv  → scaled data for Class 3')
print('   • scaler.joblib     → the scaler object (load it next class)')
print()
print('💡 Why save the scaler too?')
print('   When a NEW customer arrives in production, we need to scale them')
print('   using the SAME scaler from training. We can not re-fit it!')

---
## 🏁 We did it!

### Today's summary

| Topic | What we learned | Tool |
| --- | --- | --- |
| Why scale? | Different sizes confuse the model | — |
| Tool 1 | Mean=0, Std=1. Best default. | `StandardScaler` |
| Tool 2 | Squeeze to [0, 1]. Risky with outliers. | `MinMaxScaler` |
| Tool 3 | Uses median + IQR. Best with outliers. | `RobustScaler` |
| The big rule | Fit on TRAIN only. Transform TEST. | `train_test_split` |
| Real test | KNN improved a lot with scaling | `accuracy_score` |

### 🎓 Words to remember

- **scaling** — make all numbers fit in a similar range
- **standardization** — mean=0, std=1 (StandardScaler)
- **normalization** — squeeze to [0, 1] (MinMaxScaler)
- **mean** — average
- **std** — standard deviation = how spread out
- **median** — middle value
- **IQR** — middle 50% range (Q3 − Q1)
- **train/test split** — split data into learning set + check set
- **data leakage** — secret info from test sneaks into train (bad!)
- **KNN** — K-Nearest Neighbors, a distance-based model
- **accuracy** — % of correct predictions

### 🎯 Where we're going

- ✅ Class 1: Clean data
- ✅ Class 2: Scale numbers (today!)
- 📅 Class 3: **Categorical encoding** — turn text categories (`Yes/No`, city names) into numbers
- 📅 Class 4: Feature engineering — build NEW useful columns
- 📅 Module 4: Train the model. Predict who will leave!

### 📤 Submit

1. Save as `Module3_Class2_<YourName>.ipynb`
2. PR to your group repo at `module-3/class_2/submissions/<YourName>/`

📐 *Great work! See you in Class 3 — encoding text!*